<a href="https://colab.research.google.com/github/xyosho/Proyecto-Homologacion-ML/blob/Sprint-2/sprint2_caracteristicas_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 2 — Ingeniería de características y regresión logística
**Proyecto:** Modelo Predictivo de Segmentación de Mercado y Propensión a la Matrícula · Unicervantes

**Pedido del docente (Semana 2):** entregar *la misma tabla de datos con columnas nuevas y útiles* (por ejemplo,
"¿Aplica a Beca Mi Vocación? Sí/No" o `segmento_rendimiento_icfes`) y *un primer modelo que sirva de punto de
comparación*. El proyecto predice **solo la matrícula**; el modelo base es una **regresión logística**.

| Sección | Contenido |
|---|---|
| 1 | Datos y partición temporal (entrenamiento, validación, prueba y campaña) |
| 2 | La entrevista de admisión en texto y el modelo NLP del puntaje vocacional |
| 3 | Lo que se aprende SOLO del entrenamiento (sin fuga de datos) |
| 4 | Las 14 columnas nuevas |
| 5 | Segmentación de mercado con k-means |
| 6 | Relación de las columnas nuevas con la matrícula |
| 7 | Regresión logística y su error |

## Antes de empezar (Google Colab)
Las credenciales **no** se escriben en ninguna celda; se guardan en los **Secretos** de Colab:

1. Abra el panel **Secretos** (icono de llave, a la izquierda).
2. Cree `SUPABASE_URL` (la URL del proyecto, `https://….supabase.co`) y `SUPABASE_KEY` (la clave **publicable**,
   `sb_publishable_…`; nunca la secreta).
3. Active **Acceso del notebook** en los dos secretos.
4. Ejecute todo: *Entorno de ejecución → Ejecutar todas*. Si aparece un aviso para dar acceso a los secretos, acéptelo.

Las celdas que empiezan con `%%writefile` crean en Colab una copia exacta del código del repositorio (carpeta `src/`).

In [ ]:
!pip -q install scikit-learn==1.9.0 supabase==2.33.0 xgboost==3.4.1 shap==0.54.1 lime==0.2.0.1

In [ ]:
%%writefile comun.py
"""Versión para Google Colab de src/comun.py: la configuración del proyecto (config/reglas_negocio.yaml)
viene incrustada porque Colab no tiene el repositorio. Generado por src/s06_construir_cuadernos.py."""
from pathlib import Path

RAIZ = Path.cwd()
DATA_PROCESSED = RAIZ / "data" / "processed"

_CONFIG = {'campus': {'nombre': 'Campus Unicervantes',
            'direccion': 'Calle 209 No. 104-15, vía Arrayanes, Bogotá',
            'lat': 4.790438,
            'lon': -74.066288,
            'fuente': 'Dirección: Acuerdo 002/2025 (membrete). Coordenada exacta de la placa Calle 209 # '
                      '104-15 (CL 209 104 15, lote 009140001001) en Placa Domiciliaria de Catastro Bogotá '
                      '(UAECD), consultada 23-oct-2026. Antes se usaba una coordenada aproximada de '
                      'OpenStreetMap (4.7895, -74.0660), a unos 100 m.'},
 'tarifas': {'AD': {2024: {'valor': 4419000,
                           'fuente': 'Precio de matrícula según recibos de pago: orden de pago Pregrado '
                                     "2024-2, concepto 'MAT ING ANALITICA DE DATOS NUEVOS'."},
                    2025: {'valor': 4658000,
                           'fuente': 'Precio de matrícula según recibos de pago 2025-1 y 2025-2: $3.260.600 '
                                     'con beca del 30% => 3.260.600 / 0,70; coincide con la versión 2025 de '
                                     'la página del programa.'},
                    2026: {'valor': 4914000,
                           'fuente': 'Acuerdo 002 del 11 de noviembre de 2025 del Consejo Directivo, art. 2 '
                                     '(valores de matrícula 2026).'},
                    2027: {'valor': 4914000,
                           'fuente': 'supuesto: S-01 (se mantiene el valor vigente hasta que se publique el '
                                     'acuerdo 2027).'}},
             'SSIA': {2026: {'valor': 4950000,
                             'fuente': 'Oferta académica oficial '
                                       'unicervantes.edu.co/admision/oferta-academica (consultada '
                                       '19-oct-2026).'},
                      2027: {'valor': 4950000, 'fuente': 'supuesto: S-01.'}}},
 'becas': {'SABER_11': {'nombre': 'Beca Saber 11',
                        'umbral_icfes': 310,
                        'pct_primer_periodo': 0.35,
                        'pct_siguientes': 0.3,
                        'promedio_minimo': 4.0,
                        'fuente': 'Admisiones (corrección del 31-oct-2026): 35% el 1er semestre y 30% desde '
                                  'el 2º; coincide con los recibos (2024-2: 35%; 2025 y 2026: 30%). Página '
                                  "oficial: 'Puntaje igual o superior a 310 ... 30% para toda la carrera por "
                                  "promedio académico'."},
           'MI_VOCACION': {'nombre': 'Beca Cervantes #MiVocación',
                           'pct_primer_periodo': 0.3,
                           'pct_siguientes': 0.2,
                           'promedio_minimo': 4.0,
                           'fuente': "Admisiones ('30% para el primer periodo académico y en adelante según "
                                     "la tabla relacionada'; se mantiene con promedio acumulado >= 4.0 y se "
                                     "pierde por sanciones); landing del programa ('30% 1er semestre y 20% "
                                     "el resto de la carrera')."},
           'NINGUNA': {'nombre': 'Sin beca',
                       'pct_primer_periodo': 0.0,
                       'pct_siguientes': 0.0,
                       'promedio_minimo': None},
           'acumulables': False},
 'particion': {'entrenamiento': ['2024-1', '2024-2', '2025-1', '2025-2'],
               'validacion': ['2026-1'],
               'prueba': ['2026-2'],
               'campana': ['2027-1']},
 'caracteristicas': {'bandas_icfes': [{'desde': 0, 'hasta': 250, 'segmento': 'BAJO'},
                                      {'desde': 250, 'hasta': 310, 'segmento': 'MEDIO'},
                                      {'desde': 310, 'hasta': 350, 'segmento': 'ALTO'},
                                      {'desde': 350, 'hasta': 501, 'segmento': 'SUPERIOR'}],
                     'zonas_distancia_km': {'CERCA': 10, 'MEDIA': 20},
                     'canal_agrupado': {'EVENTO': ['EXPOESTUDIANTES', 'CAMPUS_DAY'],
                                        'RELACIONAL': ['REFERIDO', 'INMERSION', 'COLEGIO_CONVENIO'],
                                        'DIGITAL': ['SITIO_WEB', 'REDES_SOCIALES', 'LLAMADA_WHATSAPP']},
                     'estrato_grupo': {1: 'BAJO', 2: 'BAJO', 3: 'MEDIO', 4: 'ALTO', 5: 'ALTO', 6: 'ALTO'},
                     'kmeans': {'k_min': 3,
                                'k_max': 7,
                                'tolerancia_silueta': 0.01,
                                'semilla': 20261029,
                                'variables': ['estrato',
                                              'log_distancia',
                                              'punt_global',
                                              'edad_al_inscribirse',
                                              'colegio_privado']},
                     'nlp': {'ngramas': [1, 2],
                             'min_df': 3,
                             'alphas': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0],
                             'pliegues': 5,
                             'semilla': 20261112}},
 'modelado': {'semilla': 20261105,
              'umbral_clasificacion': 0.5,
              'segmentos_propension': {'ALTA': 0.6, 'BAJA': 0.35},
              'sprint2_logistica': {'C': 1.0},
              'grillas': {'logistica': {'C': [0.01, 0.1, 1.0, 10.0],
                                        'l1_ratio': [0.0, 1.0],
                                        'solver': ['saga']},
                          'random_forest': {'max_depth': [4, 6, 8, None],
                                            'min_samples_leaf': [5, 20],
                                            'max_features': ['sqrt', 0.5]},
                          'xgboost': {'n_estimators': [200, 400],
                                      'max_depth': [2, 3, 4],
                                      'learning_rate': [0.03, 0.1],
                                      'min_child_weight': [5]}}},
 'impacto': {'seguimiento': {'capacidad_pct': 0.3, 'efecto_maximo': 0.1, 'costo_por_inscrito': 30000},
             'expo': {'inversion_evento': 22000000}},
 'escala_desempeno_bachillerato': {'SUPERIOR': 4.6, 'ALTO': 4.0, 'BASICO': 3.0},
 'fondos_educacion_municipal': {'FOES_CHIA': {'ubicacion_id': 25175,
                                              'nombre': 'FOES - Fondo para el Fomento de la Educación '
                                                        'Superior de Chía',
                                              'norma': 'Acuerdo 172 de 2020; Resolución 2114 de 2021',
                                              'convenio_unicervantes': 'VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Chía (certificado de la JAC), '
                                                            'registro en el SISBÉN de Chía, certificado de '
                                                            'notas, Saber 11 y recibo de matrícula',
                                              'renovacion': 'Cada semestre (abril a octubre) según el '
                                                            'promedio del semestre anterior fijado en el '
                                                            'convenio',
                                              'fuente': 'Alcaldía de Chía (2022). Conozca el programa FOES: '
                                                        'Unicervantes en convenio (n.º 15); apoyo educativo '
                                                        'desde $1.000.000 en carreras presenciales.',
                                              'url': 'https://www.chia-cundinamarca.gov.co/2022/educacion/PROGRAMA%20FOES%20ACTUALIZADO%2022-09-2022.pdf'},
                                'FOES_SOPO': {'ubicacion_id': 25758,
                                              'nombre': 'FOES - Fondo Especial para el Fomento de la '
                                                        'Educación de Sopó',
                                              'norma': 'No publicada en la página consultada',
                                              'convenio_unicervantes': 'NO_VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Sopó, admitido o matriculado en un '
                                                            'programa de educación superior',
                                              'renovacion': 'No publicada',
                                              'fuente': 'Alcaldía de Sopó (2022, actualizado 2026). FOES: '
                                                        'apoyo de matrícula, transporte y sostenimiento. '
                                                        'Monto: supuesto S-31.',
                                              'url': 'https://www.sopo-cundinamarca.gov.co/educacion/publicaciones/290/foes-fondo-especial-para-el-fomento-de-la-educacion/'},
                                'FOEST_TOCANCIPA': {'ubicacion_id': 25817,
                                                    'nombre': 'FOEST - Fondo Especial de Educación Superior '
                                                              'de Tocancipá',
                                                    'norma': 'Aprobado por el Concejo Municipal (acuerdo no '
                                                             'consultado)',
                                                    'convenio_unicervantes': 'NO_VERIFICADO',
                                                    'apoyo_semestral_cop': 1000000,
                                                    'requisitos': 'Residente de Tocancipá; becas para '
                                                                  'mejores puntajes Saber 11, artistas y '
                                                                  'deportistas; descuentos por convenio',
                                                    'renovacion': 'No publicada',
                                                    'fuente': 'Alcaldía de Tocancipá (2017). $1.750 millones '
                                                              'para la educación superior. Monto por '
                                                              'estudiante: supuesto S-31.',
                                                    'url': 'https://www.tocancipa-cundinamarca.gov.co/sala-prensa/noticias-1/1-750-millones-para-educacion-superior-tocancipa'}},
 'inversion_captacion': {'por_evento': {'EXPOESTUDIANTES': 22000000, 'CAMPUS_DAY': 6000000},
                         'por_periodo': {'SITIO_WEB': {'1': 9000000, '2': 6000000},
                                         'REDES_SOCIALES': {'1': 18000000, '2': 10000000},
                                         'LLAMADA_WHATSAPP': {'1': 8000000, '2': 6000000},
                                         'COLEGIO_CONVENIO': {'1': 7000000, '2': 4000000},
                                         'INMERSION': {'1': 5000000, '2': 5000000},
                                         'REFERIDO': {'1': 3000000, '2': 3000000}},
                         'ventana_campana': {'1': ['03-01', '01-25'], '2': ['01-10', '07-25']}},
 'generador': {'matricula': {'muestra_ejemplo': {'aspirantes': 76,
                                                 'matriculados': 43,
                                                 'por_periodo': {'2024-1': [22, 16],
                                                                 '2024-2': [9, 4],
                                                                 '2025-1': [8, 6],
                                                                 '2025-2': [9, 3],
                                                                 '2026-1': [28, 14]}},
                             'beta_descuento': 3.0,
                             'beta_estrato': {1: -0.5, 2: -0.15, 3: 0.0, 4: 0.1, 5: 0.0, 6: -0.3},
                             'beta_log_distancia': -0.35,
                             'beta_icfes_cuadratico': -0.25,
                             'beta_canal': {'EXPOESTUDIANTES': -0.3,
                                            'SITIO_WEB': 0.0,
                                            'REDES_SOCIALES': -0.1,
                                            'REFERIDO': 0.5,
                                            'CAMPUS_DAY': 0.3,
                                            'LLAMADA_WHATSAPP': -0.2,
                                            'INMERSION': 0.8,
                                            'COLEGIO_CONVENIO': 0.3},
                             'beta_entrevista': 0.6,
                             'beta_programa_ssia': 0.1,
                             'beta_periodo_2': -0.1,
                             'beta_foes': 0.5},
               'entrevista_texto': {'secciones': {'INTERES_PROGRAMA': 'Interés por el programa académico',
                                                  'HISTORIA_VIDA': 'Historia de vida',
                                                  'INTERES_UNIVERSIDAD': 'Interés por la universidad'},
                                    'preguntas': [{'id': 1,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Por qué quiere estudiar esta carrera '
                                                            'profesional?'},
                                                  {'id': 2,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': 'Por favor defina su programa elegido'},
                                                  {'id': 3,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Qué es lo que más le gusta de la carrera?'},
                                                  {'id': 4,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Cómo se ve en un futuro como profesional?'},
                                                  {'id': 5,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': 'Si tuviera que elegir otra carrera, ¿cuál '
                                                            'sería? ¿Por qué?'},
                                                  {'id': 6,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': 'Cuéntenos sobre usted'},
                                                  {'id': 7,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': 'Ocupación de las personas con quien vive'},
                                                  {'id': 8,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Trabaja? ¿Cómo ha planeado financiar sus '
                                                            'estudios?'},
                                                  {'id': 9,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Cuáles considera que son sus cualidades y/o '
                                                            'habilidades? Mencionar mínimo 3'},
                                                  {'id': 10,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Qué aspectos considera que debe mejorar en '
                                                            'usted mismo? Mencionar mínimo 3'},
                                                  {'id': 11,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Cómo fue su rendimiento académico en el '
                                                            'colegio? (Asignaturas de mayor y menor '
                                                            'interés)'},
                                                  {'id': 12,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Qué le gusta hacer en sus tiempos libres?'},
                                                  {'id': 13,
                                                   'seccion': 'INTERES_UNIVERSIDAD',
                                                   'texto': '¿Por qué quiere estudiar en Unicervantes?'}],
                                    'peso_intencion': 0.4,
                                    'de_vocacion': 1.0,
                                    'de_respuesta': 0.8,
                                    'corte_nivel': 0.6,
                                    'dias_hasta_entrevista': {'min': 1, 'max': 10}}}}


def cargar_config() -> dict:
    return _CONFIG


In [ ]:
%%writefile conexion_supabase.py
"""Conexión SEGURA de Python con Supabase para leer la tabla plana dataset_matricula.

Las credenciales NUNCA se escriben en el código ni se imprimen:
  - En el computador: archivo .env en la raíz del proyecto, excluido por .gitignore (plantilla: .env.example).
    Se puede crear con:  python src/s00_configurar_credenciales.py
  - En Google Colab: panel "Secretos" (icono de llave) con SUPABASE_URL y SUPABASE_KEY, con acceso al cuaderno.

SUPABASE_KEY es la clave PUBLICABLE del proyecto. En la base, la seguridad a nivel de fila (RLS) solo le permite
LEER dataset_matricula (sin datos personales); las otras 25 tablas están cerradas y nadie puede escribir con ella.
Así, aunque la clave se filtrara, solo se podría leer el dataset anonimizado.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

TABLA = "dataset_matricula"
VARIABLES = ("SUPABASE_URL", "SUPABASE_KEY")
TAMANO_PAGINA = 1000          # la API de Supabase entrega como máximo 1.000 filas por petición
TIEMPO_LIMITE_S = 30          # si la red no responde, se desiste en vez de quedarse esperando


def en_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def obtener_credenciales() -> tuple[str, str]:
    """Devuelve (url, clave) desde los Secretos de Colab o desde .env / variables de entorno.

    En Colab, la primera vez que un cuaderno lee un secreto aparece un aviso para darle acceso: hay que aceptarlo
    (mientras no se acepte, la celda espera). Los errores explican qué falta, nunca muestran los valores.
    """
    problemas = []
    if en_colab():
        from google.colab import userdata
        valores = {}
        for v in VARIABLES:
            try:
                valores[v] = (userdata.get(v) or "").strip()
            except Exception as e:                  # SecretNotFoundError / NotebookAccessError
                valores[v] = ""
                motivo = ("el cuaderno no tiene acceso: active 'Acceso del notebook'" if "Access" in type(e).__name__
                          else "no existe un secreto con ese nombre exacto")
                problemas.append(f"{v}: {motivo}")
        donde = "los Secretos de Colab (icono de llave, con 'Acceso del notebook' activado)"
    else:
        from dotenv import load_dotenv
        load_dotenv(Path(__file__).resolve().parents[1] / ".env")
        valores = {v: os.environ.get(v, "").strip() for v in VARIABLES}
        donde = "el archivo .env (python src/s00_configurar_credenciales.py o copia de .env.example)"
    problemas += [f"{v}: está vacío" for v, x in valores.items() if not x and not any(p.startswith(v) for p in problemas)]
    url, clave = valores["SUPABASE_URL"], valores["SUPABASE_KEY"]
    if url and not (url.startswith("https://") and url.rstrip("/").endswith(".supabase.co")):
        problemas.append("SUPABASE_URL: no es la URL del proyecto (debe ser https://<proyecto>.supabase.co; "
                         "no la cadena postgresql:// del pooler)")
    if clave.lower().startswith(("postgres://", "postgresql://")):
        problemas.append("SUPABASE_KEY: es la cadena del pooler; aquí va la clave PUBLICABLE (sb_publishable_...)")
    elif clave.startswith("sb_secret_"):
        problemas.append("SUPABASE_KEY: es la clave SECRETA; use la PUBLICABLE (sb_publishable_...)")
    if problemas:
        raise RuntimeError(f"Credenciales incompletas en {donde}: " + "; ".join(problemas) + ".")
    return url, clave


def crear_cliente(url: str | None = None, clave: str | None = None):
    """Cliente de la API de Supabase con tiempo límite. La clave se usa en memoria y no se muestra."""
    from supabase import ClientOptions, create_client
    if url is None or clave is None:
        url, clave = obtener_credenciales()
    return create_client(url, clave, options=ClientOptions(postgrest_client_timeout=TIEMPO_LIMITE_S))


def leer_dataset_matricula(cliente, tabla: str = TABLA) -> pd.DataFrame:
    """Lee la tabla completa en páginas de 1.000 filas (límite de la API), ordenada por inscripcion_id."""
    filas, inicio = [], 0
    while True:
        r = (cliente.table(tabla).select("*").order("inscripcion_id")
             .range(inicio, inicio + TAMANO_PAGINA - 1).execute())
        filas += r.data
        if len(r.data) < TAMANO_PAGINA:
            break
        inicio += TAMANO_PAGINA
    return pd.DataFrame(filas)


def probar_solo_lectura(cliente, tabla: str = TABLA) -> bool:
    """Intenta modificar la tabla con la clave publicable. Devuelve True si la base lo RECHAZA por permisos
    (código 42501 de PostgreSQL: permiso denegado). La fila -1 no existe, así que nada cambiaría de todos modos."""
    try:
        cliente.table(tabla).update({"punt_global": 0}).eq("inscripcion_id", -1).execute()
    except Exception as e:                           # APIError de la API de Supabase
        return getattr(e, "code", None) == "42501"
    return False


In [ ]:
%%writefile datos.py
"""Carga de las tablas planas para los sprints 2 a 6.

Fuente preferida: Supabase con la API de solo lectura (credenciales en .env o en los Secretos de Colab, ver
conexion_supabase.py). Si no hay credenciales, se usan los CSV que exporta el pipeline (data/processed), que son
idénticos (las pruebas lo comprueban). La función siempre dice de dónde salieron los datos.
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

TABLAS = ("dataset_matricula", "dataset_entrevista", "resultado_campana_2027_1")   # el último solo en el Sprint 5
ENTEROS_CON_VACIOS = ("matriculado", "puntaje_vocacional", "matriculado_real")


def preparar_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """La API devuelve JSON: se fijan los tipos para que local y Colab den exactamente lo mismo."""
    if df.empty:                                   # tabla sin filas: no hay tipos que fijar
        return df
    df = df.copy()
    for c in ENTEROS_CON_VACIOS:
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype("Int64")
    for c in ("distancia_campus_km", "distancia_vial_km", "colegio_promedio_saber11", "pct_descuento_inicial"):
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype(float)
    if "asesor_id" in df:
        df["asesor_id"] = df["asesor_id"].astype(int)
    return df.sort_values("inscripcion_id").reset_index(drop=True)


def _csv_local(tabla: str) -> Path:
    from comun import DATA_PROCESSED
    if tabla == "resultado_campana_2027_1":       # "sobre sellado" desde la Semana 1 (data/processed/oculto)
        return DATA_PROCESSED / "oculto" / "resultado_2027_1_sellado.parquet"
    return DATA_PROCESSED / f"{tabla}.csv"


def cargar(tabla: str, fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """Devuelve (datos, origen). fuente: 'supabase', 'local' o 'auto' (Supabase si hay credenciales)."""
    if tabla not in TABLAS:
        raise ValueError(f"Tabla desconocida: {tabla}")
    if fuente in ("auto", "supabase"):
        try:
            from conexion_supabase import crear_cliente, leer_dataset_matricula
            df = leer_dataset_matricula(crear_cliente(), tabla=tabla)
            if df.empty:                       # la tabla existe pero no tiene filas (o la API no deja leerlas)
                raise RuntimeError(f"La tabla {tabla} está vacía en Supabase: cárguela con python src/s05_cargar_supabase.py.")
            return preparar_tipos(df), "Supabase (API de solo lectura)"
        except Exception as error:
            if fuente == "supabase":
                raise
            aviso = type(error).__name__
    ruta = _csv_local(tabla)
    df = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta)
    origen = "CSV local exportado por el pipeline" + (f" (sin conexión a Supabase: {aviso})" if fuente == "auto" else "")
    return preparar_tipos(df), origen


def cargar_inscritos(fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """dataset_matricula con las respuestas de la entrevista (dataset_entrevista, texto por sección) unidas por inscripcion_id."""
    inscritos, origen = cargar("dataset_matricula", fuente)
    entrevista, _ = cargar("dataset_entrevista", fuente)
    if len(entrevista) != len(inscritos):     # p. ej. la tabla existe en Supabase pero aún no se cargó (src/s05)
        if fuente == "supabase":
            raise RuntimeError(f"dataset_entrevista tiene {len(entrevista)} filas y dataset_matricula {len(inscritos)}: "
                               "falta cargar la entrevista en Supabase (python src/s05_cargar_supabase.py).")
        entrevista, _ = cargar("dataset_entrevista", "local")
    return inscritos.merge(entrevista, on="inscripcion_id", how="left", validate="one_to_one"), origen


In [ ]:
%%writefile nlp_vocacion.py
"""SPRINT 2 — Modelo NLP: el puntaje vocacional que se lee en las palabras del aspirante.

Entrada: dataset_entrevista, las respuestas abiertas de la entrevista de admisión unidas por sección
(Interés por el programa académico, Historia de vida e Interés por la universidad).
Objetivo: el puntaje vocacional que dio el entrevistador (1 a 5).
Modelo: TF-IDF por sección (palabras y pares de palabras, sin tildes ni palabras vacías) + regresión Ridge.

El resultado, puntaje_vocacional_nlp (1 a 5), es la vocación según las palabras del propio aspirante. No es una copia
del entrevistador: las palabras también reflejan lo que él no alcanza a ver (supuesto S-49), y eso es lo que la
regresión logística del Sprint 3 puede aprovechar.

Regla contra la FUGA DE DATOS: se ajusta SOLO con el ENTRENAMIENTO. Las filas de entrenamiento reciben predicciones
fuera de pliegue (validación cruzada de 5 pliegues): ninguna se puntúa con un modelo que ya vio su propia respuesta.

Funciones: ajustar(entrenamiento) -> dict · puntuar(nlp, tabla) -> Series · palabras_clave(nlp) · metricas(y, p)
"""
from __future__ import annotations

import re
import unicodedata
from functools import partial

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline

SECCIONES = {"texto_interes_programa": "Interés por el programa académico",
             "texto_historia_vida": "Historia de vida",
             "texto_interes_universidad": "Interés por la universidad"}

# Palabras vacías del español, sin tildes (el TF-IDF quita las tildes antes de compararlas). Se CONSERVAN las que
# cambian el sentido de una respuesta: "no", "nunca", "todavia", "ya", "muy", "sin" y las muletillas de duda ("pues").
STOPWORDS_ES = sorted({
    "a", "al", "algo", "algun", "alguna", "algunas", "alguno", "algunos", "ante", "antes", "asi", "aun", "cada",
    "como", "con", "contra", "cual", "cuales", "cuando", "de", "del", "desde", "donde", "durante", "e", "el", "ella",
    "ellas", "ellos", "en", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba",
    "estas", "este", "esto", "estos", "fue", "fueron", "ha", "han", "hasta", "hay", "la", "las", "le", "les", "lo",
    "los", "me", "mi", "mis", "nos", "nosotros", "o", "os", "otra", "otras", "otro", "otros", "para", "pero", "por",
    "que", "quien", "quienes", "se", "ser", "si", "sobre", "son", "su", "sus", "te", "ti", "tu", "tus", "u",
    "un", "una", "unas", "uno", "unos", "y", "yo"})


def configuracion() -> dict:
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


def analizar(texto: str, ngramas: tuple = (1, 2)) -> list[str]:
    """Cómo ve el modelo una respuesta: minúsculas y sin tildes, se parte en frases (cada respuesta termina en punto),
    se quitan las palabras vacías y se forman palabras y pares de palabras DENTRO de cada frase. Así un par nunca
    mezcla el final de una respuesta con el inicio de la siguiente."""
    t = "".join(c for c in unicodedata.normalize("NFKD", str(texto).lower()) if not unicodedata.combining(c))
    salida = []
    for frase in re.split(r"[.;:!?¡¿]+", t):
        palabras = [w for w in re.findall(r"\b\w\w+\b", frase) if w not in STOPWORDS_ES]
        for n in range(ngramas[0], ngramas[1] + 1):
            salida += [" ".join(palabras[i:i + n]) for i in range(len(palabras) - n + 1)]
    return salida


def _pipeline(cfg: dict) -> Pipeline:
    n = cfg["caracteristicas"]["nlp"]
    analizador = partial(analizar, ngramas=tuple(n["ngramas"]))
    tfidf = ColumnTransformer([(col, TfidfVectorizer(analyzer=analizador, min_df=n["min_df"], sublinear_tf=True), col)
                               for col in SECCIONES])
    return Pipeline([("tfidf", tfidf), ("ridge", RidgeCV(alphas=n["alphas"]))])


def con_texto(d: pd.DataFrame) -> pd.Series:
    """True si el inscrito presentó la entrevista (tiene respuestas)."""
    if not set(SECCIONES) <= set(d.columns):
        return pd.Series(False, index=d.index)
    return d[list(SECCIONES)].notna().all(axis=1)


def ajustar(entrenamiento: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, solo con las entrevistas del ENTRENAMIENTO, a leer el puntaje vocacional en las palabras."""
    cfg = cfg or configuracion()
    n = cfg["caracteristicas"]["nlp"]
    e = entrenamiento[con_texto(entrenamiento)]
    X, y = e[list(SECCIONES)], e["puntaje_vocacional"].astype(float).to_numpy()
    modelo = _pipeline(cfg).fit(X, y)
    pliegues = KFold(n["pliegues"], shuffle=True, random_state=n["semilla"])
    fuera = np.clip(cross_val_predict(_pipeline(cfg), X, y, cv=pliegues), 1, 5)
    return {"modelo": modelo, "fuera_de_pliegue": pd.Series(fuera, index=e["inscripcion_id"].to_numpy()),
            "alpha": float(modelo.named_steps["ridge"].alpha_), "n_entrevistas": int(len(e)),
            "n_terminos": int(len(modelo.named_steps["tfidf"].get_feature_names_out())), "media_entrenamiento": float(y.mean())}


def puntuar(nlp: dict, d: pd.DataFrame) -> pd.Series:
    """puntaje_vocacional_nlp (1 a 5) de cada fila; vacío si no hubo entrevista. Las filas del entrenamiento usan su
    predicción fuera de pliegue; las demás (validación, prueba, campaña y aspirantes nuevos), el modelo completo."""
    s = pd.Series(np.nan, index=d.index, dtype=float)
    m = con_texto(d)
    if m.any():
        s[m] = np.clip(nlp["modelo"].predict(d.loc[m, list(SECCIONES)]), 1, 5)
        fuera = d["inscripcion_id"].map(nlp["fuera_de_pliegue"])
        s = fuera.where(fuera.notna() & m, s)
    return s.round(3)


def terminos(texto: str, cfg: dict | None = None) -> list[str]:
    """Los términos que el modelo extrae de un texto (para mostrar el preprocesamiento)."""
    return analizar(texto, tuple((cfg or configuracion())["caracteristicas"]["nlp"]["ngramas"]))


def palabras_clave(nlp: dict, k: int = 8) -> pd.DataFrame:
    """Los términos que más suben y más bajan el puntaje en cada sección: las palabras que determinan la vocación."""
    nombres = nlp["modelo"].named_steps["tfidf"].get_feature_names_out()
    pesos = pd.DataFrame({"columna": [x.split("__", 1)[0] for x in nombres], "termino": [x.split("__", 1)[1] for x in nombres],
                          "peso": nlp["modelo"].named_steps["ridge"].coef_})
    filas = []
    for col, nombre in SECCIONES.items():
        p = pesos[pesos["columna"] == col]
        for sentido, parte in (("sube", p.nlargest(k, "peso")), ("baja", p.nsmallest(k, "peso"))):
            filas += [{"seccion": nombre, "sentido": sentido, "termino": t, "peso": round(float(w), 4)}
                      for t, w in zip(parte["termino"], parte["peso"])]
    return pd.DataFrame(filas)


def metricas(y, p, media_entrenamiento: float | None = None) -> dict:
    """Qué tan bien lee el modelo el puntaje del entrevistador (y frente a predecir siempre el promedio)."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    base = float(np.mean(y) if media_entrenamiento is None else media_entrenamiento)
    return {"n": int(len(y)), "mae": float(np.mean(np.abs(y - p))), "mae_promedio": float(np.mean(np.abs(y - base))),
            "spearman": float(spearmanr(y, p).statistic), "pct_a_un_punto": float(100 * np.mean(np.abs(y - p) <= 1))}


In [ ]:
%%writefile caracteristicas.py
"""SPRINT 2 — Ingeniería de características: columnas nuevas y útiles sobre la tabla plana dataset_matricula.

Regla de oro contra la FUGA DE DATOS: todo lo que se APRENDE de los datos (las medias para estandarizar, los grupos
de k-means y el modelo NLP de la entrevista) se ajusta SOLO con el conjunto de ENTRENAMIENTO
(2024-1 a 2025-2) y después se aplica igual a validación, prueba, campaña y a cualquier aspirante nuevo (Sprint 6).
Si se ajustara con todos los datos, el modelo "vería" información del futuro y sus métricas serían optimistas.

Funciones principales:
  particion_temporal(periodo)                      -> ENTRENAMIENTO / VALIDACION / PRUEBA / CAMPANA
  ajustar_referencias(inscritos)                   -> lo aprendido del entrenamiento (un diccionario)
  construir_caracteristicas(inscritos, referencias) -> la misma tabla con las columnas nuevas
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import nlp_vocacion as nlp

NUEVAS = [
    "particion", "aplica_beca_saber11", "aplica_beca_mi_vocacion", "puntaje_vocacional_nlp",
    "segmento_rendimiento_icfes",
    "ahorro_beca_primer_semestre", "log_distancia", "zona_distancia", "rezago_saber11",
    "canal_agrupado", "estrato_grupo", "colegio_privado", "segmento_mercado",
]

# Descripción de cada columna nueva (diccionario del Sprint 2)
DICCIONARIO = {
    "particion": "Conjunto temporal: ENTRENAMIENTO (2024-1 a 2025-2), VALIDACION (2026-1), PRUEBA (2026-2), CAMPANA (2027-1)",
    "aplica_beca_saber11": "Sí/No: puntaje global Saber 11 ≥ 310 (requisito de la Beca Saber 11)",
    "aplica_beca_mi_vocacion": "Sí/No: presentó entrevista y su puntaje vocacional es ≥ 3 (requisito de Mi Vocación)",
    "puntaje_vocacional_nlp": "Puntaje vocacional de 1 a 5 que el modelo NLP lee en las palabras del aspirante (respuestas de la entrevista; TF-IDF + Ridge ajustado con el entrenamiento); vacío si no hubo entrevista",
    "segmento_rendimiento_icfes": "BAJO (< 250), MEDIO (250-309), ALTO (310-349: ya aplica a beca) o SUPERIOR (≥ 350)",
    "ahorro_beca_primer_semestre": "Cuánto le ahorra la beca en el 1er semestre (tarifa plena − precio neto), en COP",
    "log_distancia": "Logaritmo natural de la distancia por vías desde la dirección hasta el campus (S-44; reduce la cola larga)",
    "zona_distancia": "Distancia por vías desde la dirección: CERCA (≤ 10 km), MEDIA (10-20 km) o LEJOS (> 20 km)",
    "rezago_saber11": "INMEDIATO (presentó el Saber 11 el año anterior), UN_ANIO o DOS_O_MAS",
    "canal_agrupado": "EVENTO (Expo, Campus Day), RELACIONAL (referido, inmersión, convenio) o DIGITAL (web, redes, llamada)",
    "estrato_grupo": "BAJO (1-2), MEDIO (3) o ALTO (4-6)",
    "colegio_privado": "1 si el colegio es privado",
    "segmento_mercado": "Perfil del aspirante según k-means (segmentación de mercado), nombrado por sus rasgos principales",
}

ETIQUETAS_KMEANS = {   # (si el centro está por encima, si está por debajo) de la media del entrenamiento
    "estrato": ("estrato más alto", "estrato más bajo"),
    "log_distancia": ("viven lejos del campus", "viven cerca del campus"),
    "punt_global": ("buen puntaje ICFES", "puntaje ICFES bajo"),
    "edad_al_inscribirse": ("aspirantes mayores", "aspirantes más jóvenes"),
    "colegio_privado": ("colegio privado", "colegio público"),
    "colegio_promedio_saber11": ("colegio de alto nivel", "colegio de nivel bajo"),
}

# Nombres en español sencillo para tablas y gráficas (los nombres técnicos de las columnas no cambian)
NOMBRES_LEGIBLES = {
    "estrato": "Estrato promedio",
    "log_distancia": "Distancia por vías (km)",
    "distancia_vial_km": "Distancia por vías (km)",
    "punt_global": "Puntaje ICFES",
    "edad_al_inscribirse": "Edad (años)",
    "colegio_privado": "% de colegio privado",
    "colegio_promedio_saber11": "Nivel ICFES del colegio",
    "segmento_mercado": "Segmento de mercado",
    "segmento_rendimiento_icfes": "Puntaje ICFES",
    "aplica_beca_mi_vocacion": "¿Aplica a la beca Mi Vocación?",
    "aplica_beca_saber11": "¿Aplica a la beca Saber 11?",
    "canal_agrupado": "Canal por el que llegó",
    "zona_distancia": "Distancia al campus",
    "estrato_grupo": "Estrato",
    "rezago_saber11": "Tiempo desde el ICFES",
    "puntaje_vocacional_nlp": "Puntaje vocacional según sus palabras (NLP, 1 a 5)",
}

# El ICFES llama "oficial" al colegio público. Los datos y los modelos conservan el código OFICIAL; en todo texto que
# se muestra (explicaciones, gráficas, script de predicción) se dice "público".
NOMBRE_TIPO_COLEGIO = {"OFICIAL": "público", "PRIVADO": "privado", "TECNICO": "técnico"}


def tipo_colegio_codigo(valor: str) -> str:
    """Código de datos del tipo de colegio: acepta PUBLICO (o PÚBLICO) como sinónimo de OFICIAL."""
    v = str(valor).strip().upper().replace("Ú", "U").replace("É", "E")
    return "OFICIAL" if v == "PUBLICO" else v


def etiquetas_categorias(cfg: dict | None = None) -> dict:
    """Texto claro para los códigos de las columnas categóricas nuevas (p. ej. CERCA → "Cerca (hasta 10 km)")."""
    c = (cfg or configuracion())["caracteristicas"]
    z = c["zonas_distancia_km"]
    icfes = {b["segmento"]: (f"{b['segmento'].capitalize()} (menos de {b['hasta']})" if b["desde"] == 0 else
                             f"{b['segmento'].capitalize()} ({b['desde']} o más)" if b["hasta"] > 500 else
                             f"{b['segmento'].capitalize()} ({b['desde']} a {b['hasta'] - 1})") for b in c["bandas_icfes"]}
    return {
        "segmento_rendimiento_icfes": icfes,
        "zona_distancia": {"CERCA": f"Cerca (hasta {z['CERCA']} km)", "MEDIA": f"Media ({z['CERCA']} a {z['MEDIA']} km)",
                           "LEJOS": f"Lejos (más de {z['MEDIA']} km)"},
        "canal_agrupado": {"EVENTO": "Eventos (Expo, Campus Day)", "RELACIONAL": "Relacional (referido, convenio…)",
                           "DIGITAL": "Digital (web, redes, llamada)"},
        "estrato_grupo": {"BAJO": "Estratos 1 y 2", "MEDIO": "Estrato 3", "ALTO": "Estratos 4 a 6"},
        "rezago_saber11": {"INMEDIATO": "Sin pausa tras el colegio", "UN_ANIO": "1 año de pausa", "DOS_O_MAS": "2 años o más de pausa"},
        "aplica_beca_mi_vocacion": {"Sí": "Sí aplica", "No": "No aplica"},
        "aplica_beca_saber11": {"Sí": "Sí aplica", "No": "No aplica"},
    }


def tabla_segmentos(d: pd.DataFrame) -> pd.DataFrame:
    """Resumen de cada segmento de mercado con valores reales (no estandarizados), listo para leer o presentar."""
    m = d.assign(matriculado=d["matriculado"].astype(float))
    def resumen(g: pd.DataFrame) -> dict:
        return {"Inscritos": len(g), "% del total": len(g) / len(d) * 100,
                "% que se matriculó": g["matriculado"].mean() * 100, "Puntaje ICFES": g["punt_global"].mean(),
                "Distancia por vías (km)": g["distancia_vial_km"].mean(), "Estrato promedio": g["estrato"].mean(),
                "% de colegio público": (1 - g["colegio_privado"].mean()) * 100, "Edad (años)": g["edad_al_inscribirse"].mean()}
    t = pd.DataFrame({s: resumen(g) for s, g in m.groupby("segmento_mercado")}).T
    t.loc["Todos los inscritos"] = resumen(m)
    t.index.name = "Segmento de mercado"
    return t.round({"Inscritos": 0, "% del total": 1, "% que se matriculó": 1, "Puntaje ICFES": 0,
                    "Distancia por vías (km)": 1, "Estrato promedio": 1,
                    "% de colegio público": 0, "Edad (años)": 1}).astype({"Inscritos": int})


def configuracion() -> dict:
    """Parámetros del proyecto: config/reglas_negocio.yaml en el computador; en Colab, la copia del cuaderno."""
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


# ----------------------------------------------------------------------------- reglas (no se aprenden de datos)
def particion_temporal(periodo: pd.Series, cfg: dict | None = None) -> pd.Series:
    p = (cfg or configuracion())["particion"]
    mapa = {per: nombre.upper() for nombre in ("entrenamiento", "validacion", "prueba", "campana") for per in p[nombre]}
    return periodo.map(mapa)


def segmento_icfes(punt_global: pd.Series, cfg: dict | None = None) -> pd.Series:
    bandas = (cfg or configuracion())["caracteristicas"]["bandas_icfes"]
    cortes = [bandas[0]["desde"]] + [b["hasta"] for b in bandas]
    return pd.cut(punt_global, bins=cortes, right=False, labels=[b["segmento"] for b in bandas]).astype(str)


def _reglas(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    c = cfg["caracteristicas"]
    d = df.copy()
    d["particion"] = particion_temporal(d["periodo"], cfg)
    d["aplica_beca_saber11"] = np.where(d["punt_global"] >= cfg["becas"]["SABER_11"]["umbral_icfes"], "Sí", "No")
    d["aplica_beca_mi_vocacion"] = np.where(d["cumple_req_mi_vocacion"] == 1, "Sí", "No")
    d["segmento_rendimiento_icfes"] = segmento_icfes(d["punt_global"], cfg)
    # El precio que se usa es el neto del 1er semestre (valor_neto_primer_semestre, ya en dataset_matricula)
    d["ahorro_beca_primer_semestre"] = d["valor_semestre_bruto"] - d["valor_neto_primer_semestre"]
    vial = d["distancia_vial_km"].astype(float)                 # desde la dirección del formulario (S-44)
    d["log_distancia"] = np.log(np.maximum(vial, 0.5))           # piso de 0,5 km: un vecino del campus no da −∞
    z = c["zonas_distancia_km"]
    d["zona_distancia"] = np.select([vial <= z["CERCA"], vial <= z["MEDIA"]],
                                    ["CERCA", "MEDIA"], "LEJOS")
    d["rezago_saber11"] = np.select([d["anios_desde_saber11"] == 0, d["anios_desde_saber11"] == 1],
                                    ["INMEDIATO", "UN_ANIO"], "DOS_O_MAS")
    canal = {canal: grupo for grupo, canales in c["canal_agrupado"].items() for canal in canales}
    d["canal_agrupado"] = d["canal"].map(canal)
    d["estrato_grupo"] = d["estrato"].map(lambda e: c["estrato_grupo"][int(e)])
    d["colegio_privado"] = (d["colegio_naturaleza"] == "PRIVADO").astype(int)
    return d


# ----------------------------------------------------------------------------- lo que se aprende del entrenamiento
def ajustar_referencias(inscritos: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, SOLO con las filas de ENTRENAMIENTO, todo lo que necesitan las columnas nuevas."""
    cfg = cfg or configuracion()
    ent = _reglas(inscritos, cfg)
    ent = ent[ent["particion"] == "ENTRENAMIENTO"]
    ref: dict = {"periodos_entrenamiento": sorted(ent["periodo"].unique()), "n_entrenamiento": len(ent)}

    # 1) Estandarización (z) del Saber 11 con la media y la desviación del entrenamiento (término cuadrático del modelo)
    ref["estandarizacion"] = {c: (float(ent[c].mean()), float(ent[c].std())) for c in ["punt_global"]}

    # 2) Segmentación de mercado con k-means: se prueba k = k_min..k_max; se elige el MENOR k cuya silueta esté a
    #    menos de tolerancia_silueta de la mejor (parsimonia: menos segmentos son más fáciles de usar para Mercadeo)
    km = cfg["caracteristicas"]["kmeans"]
    variables = km["variables"]
    medias, desv = ent[variables].mean(), ent[variables].std()
    Z = ((ent[variables] - medias) / desv).to_numpy(float)
    evaluacion = []
    for k in range(km["k_min"], km["k_max"] + 1):
        modelo = KMeans(n_clusters=k, n_init=20, random_state=km["semilla"]).fit(Z)
        evaluacion.append({"k": k, "inercia": float(modelo.inertia_),
                           "silueta": float(silhouette_score(Z, modelo.labels_, random_state=km["semilla"]))})
    tope = max(e["silueta"] for e in evaluacion) - km.get("tolerancia_silueta", 0.0)
    mejor = min((e for e in evaluacion if e["silueta"] >= tope), key=lambda e: e["k"])
    modelo = KMeans(n_clusters=mejor["k"], n_init=20, random_state=km["semilla"]).fit(Z)
    orden = np.argsort(-np.bincount(modelo.labels_))            # segmento 1 = el más grande
    centros = pd.DataFrame(modelo.cluster_centers_[orden], columns=variables)
    nombres = []
    for i, fila in centros.iterrows():
        rasgos = fila.abs().sort_values(ascending=False).index[:2]
        rasgos_txt = ", ".join(ETIQUETAS_KMEANS[v][0 if fila[v] > 0 else 1] for v in rasgos)
        nombres.append(f"S{i + 1}: " + rasgos_txt[0].upper() + rasgos_txt[1:])   # "S1: Colegio público, viven cerca…"
    ref["kmeans"] = {"modelo": modelo, "orden": orden.tolist(), "variables": variables, "medias": medias, "desv": desv,
                     "evaluacion": evaluacion, "k": mejor["k"], "nombres": nombres, "centros_z": centros}

    # 3) Modelo NLP de la entrevista: lee el puntaje vocacional en las respuestas abiertas (src/nlp_vocacion.py).
    #    Las filas de entrenamiento quedan con su predicción fuera de pliegue.
    ref["nlp"] = nlp.ajustar(ent, cfg)
    return ref


def construir_caracteristicas(inscritos: pd.DataFrame, ref: dict, cfg: dict | None = None) -> pd.DataFrame:
    """Devuelve la misma tabla con las columnas nuevas, aplicando lo aprendido del entrenamiento."""
    cfg = cfg or configuracion()
    d = _reglas(inscritos, cfg)
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    etiqueta = km["modelo"].predict(Z)
    posicion = {original: nuevo for nuevo, original in enumerate(km["orden"])}
    d["segmento_mercado"] = [km["nombres"][posicion[e]] for e in etiqueta]
    d["puntaje_vocacional_nlp"] = nlp.puntuar(ref["nlp"], d)
    # El texto de la entrevista es la entrada del NLP, no una columna del modelo: no pasa a la tabla del Sprint 2
    originales = [c for c in inscritos.columns if c not in nlp.SECCIONES]
    return d[originales + [c for c in NUEVAS if c not in originales]]


In [ ]:
%%writefile modelado.py
"""Modelos y métricas de la matrícula (Sprint 2: regresión logística · Sprint 3: comparación con Random Forest y XGBoost).

El primer modelo del proyecto (Sprint 2) es una REGRESIÓN LOGÍSTICA sin ajuste de hiperparámetros
(config/reglas_negocio.yaml -> modelado.sprint2_logistica): es el modelo base del proyecto. En el Sprint 3 se ajustan
sus hiperparámetros con la validación y se compara con Random Forest y XGBoost.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score)


def metricas(y, p, umbral: float = 0.5) -> dict:
    """Métricas de clasificación. 'error_pct' es el lenguaje del docente: % de inscritos mal clasificados."""
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    pred = (p >= umbral).astype(int)
    unica = len(np.unique(y)) > 1
    return {
        "n": int(len(y)), "umbral": float(umbral),
        "error_pct": float(100 * (pred != y).mean()),
        "exactitud": float((pred == y).mean()),
        "auc_roc": float(roc_auc_score(y, p)) if unica else float("nan"),
        "auc_pr": float(average_precision_score(y, p)) if unica else float("nan"),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "tasa_real": float(y.mean()), "tasa_predicha_media": float(p.mean()),
    }


def tabla_metricas(resultados: dict[str, dict]) -> pd.DataFrame:
    """{nombre_modelo: metricas} -> tabla ordenada para mostrar."""
    cols = ["n", "error_pct", "exactitud", "auc_roc", "auc_pr", "brier", "precision", "recall", "f1"]
    return pd.DataFrame(resultados).T[cols].astype(float).round(4)


# =============================================================================
# Modelos de Machine Learning (regresión logística desde el Sprint 2; Random Forest y XGBoost en el Sprint 3)
# =============================================================================
# Variables de entrada. Todas se conocen ANTES de la decisión que se predice (sin fuga de datos).
# No se usa colegio_promedio_saber11 (media ICFES del colegio): Admisiones no suele tener ese dato. Tampoco el promedio
# del colegio: dejó de ser una variable del análisis (sus notas quedan solo en las tablas del bachillerato).
# Sprint 2 (modelo base): la entrevista entra con el puntaje vocacional del entrevistador, sin el NLP.
VARIABLES_MATRICULA = {
    "numericas": ["punt_global", "icfes_al_cuadrado", "edad_al_inscribirse", "estrato",
                  "log_distancia", "dias_registro_a_inscripcion", "contactos_antes_de_inscribirse",
                  "anios_desde_saber11", "puntaje_vocacional", "pct_descuento_inicial",
                  "valor_neto_primer_semestre"],
    "binarias": ["entrevista_realizada", "apoyo_foes", "elegible_foes", "colegio_en_convenio", "captado_en_evento",
                 "colegio_privado", "semestre_2"],
    "categoricas": ["programa", "genero", "canal", "beca", "tipo_colegio", "regimen_salud", "segmento_mercado"],
}
# Sprint 3: las mismas variables + el puntaje vocacional que el modelo NLP lee en las palabras del aspirante
VARIABLES_MATRICULA_NLP = {**VARIABLES_MATRICULA,
                           "numericas": VARIABLES_MATRICULA["numericas"] + ["puntaje_vocacional_nlp"]}


def preparar_X(d: pd.DataFrame, referencias: dict, variables: dict) -> pd.DataFrame:
    """Columnas de entrada del modelo, con el tratamiento de nulos del Sprint 1 y dos derivadas simples."""
    X = d.copy()
    # 0 = sin entrevista (entrevista_realizada distingue "sin entrevista" de un puntaje bajo)
    for c in ("puntaje_vocacional", "puntaje_vocacional_nlp"):
        if c in X:
            X[c] = X[c].astype("Float64").fillna(0).astype(float)
    media, desv = referencias["estandarizacion"]["punt_global"]
    X["icfes_al_cuadrado"] = ((X["punt_global"] - media) / desv) ** 2       # captura la "U invertida" del ICFES
    X["semestre_2"] = X["periodo"].str.endswith("-2").astype(int)          # ingreso a mitad de año
    columnas = variables["numericas"] + variables["binarias"] + variables["categoricas"]
    X = X[columnas].copy()
    for c in variables["categoricas"]:
        X[c] = X[c].astype(str)
    for c in variables["numericas"] + variables["binarias"]:
        X[c] = X[c].astype(float)
    return X


def construir_pipeline(tipo: str, parametros: dict, variables: dict, semilla: int = 20261105):
    """Preprocesamiento + modelo en un solo objeto: se ajusta y se aplica igual en todos los conjuntos."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier

    categ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if tipo == "logistica":
        prep = ColumnTransformer([("num", StandardScaler(), variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        modelo = LogisticRegression(max_iter=5000, random_state=semilla, **parametros)   # saga baraja: semilla fija
    else:
        prep = ColumnTransformer([("num", "passthrough", variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        if tipo == "random_forest":
            modelo = RandomForestClassifier(n_estimators=500, random_state=semilla, n_jobs=-1, **parametros)
        elif tipo == "xgboost":
            modelo = XGBClassifier(random_state=semilla, n_jobs=1, eval_metric="logloss", tree_method="hist",
                                   subsample=0.8, colsample_bytree=0.8, **parametros)
        else:
            raise ValueError(tipo)
    return Pipeline([("preparacion", prep), ("modelo", modelo)])


def grilla(tipo: str, cfg: dict) -> list[dict]:
    """Combinaciones de hiperparámetros a probar (config/reglas_negocio.yaml -> modelado.grillas)."""
    from itertools import product
    g = cfg["modelado"]["grillas"][tipo]
    return [dict(zip(g, valores)) for valores in product(*g.values())]


def ajustar_temporal(tipo: str, cfg: dict, Xe, ye, Xv, yv, variables: dict) -> pd.DataFrame:
    """Prueba cada combinación entrenando con ENTRENAMIENTO y midiendo en VALIDACION (2026-1)."""
    filas = []
    for params in grilla(tipo, cfg):
        modelo = construir_pipeline(tipo, params, variables, cfg["modelado"]["semilla"]).fit(Xe, ye)
        m = metricas(yv, modelo.predict_proba(Xv)[:, 1])
        filas.append({"tipo": tipo, **{f"param_{k}": v for k, v in params.items()},
                      "log_loss_validacion": m["log_loss"], "auc_validacion": m["auc_roc"],
                      "brier_validacion": m["brier"], "error_pct_validacion": m["error_pct"], "parametros": params})
    return pd.DataFrame(filas).sort_values("log_loss_validacion").reset_index(drop=True)


def intervalo_bootstrap(y, p, funcion, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% por remuestreo: se repite la medición en 1.000 muestras con reemplazo del conjunto de prueba."""
    y, p = np.asarray(y, dtype=int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(semilla)
    valores = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        valores.append(funcion(y[i], p[i], umbral))
    return float(np.percentile(valores, 2.5)), float(np.percentile(valores, 97.5))


def diferencia_error(y, p_antes, p_despues, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% de (error del modelo nuevo − error del anterior), en puntos, remuestreando los MISMOS inscritos.
    Si el intervalo incluye el 0, la diferencia puede ser azar: los dos modelos rinden igual."""
    y = np.asarray(y, dtype=int)
    falla_a = (np.asarray(p_antes) >= umbral) != y
    falla_b = (np.asarray(p_despues) >= umbral) != y
    rng = np.random.default_rng(semilla)
    dif = [100 * (falla_b[i].mean() - falla_a[i].mean()) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(dif, 2.5)), float(np.percentile(dif, 97.5))


def frase_docente(error_antes: float, error_despues: float, nombre: str, ic: tuple[float, float], n: int,
                  antes: str = "La regresión logística de la Semana 2", cuando: str = "inscritos de 2026-2") -> str:
    """La frase que pide el docente ("tenía un error del X%, este modelo del Y%"), sin afirmar una mejora que no hay."""
    num = lambda v: f"{v:.1f}".replace(".", ",")   # noqa: E731
    frase = (f"{antes} tenía un error del {num(error_antes)}%, mientras que el modelo de la Semana 3 ({nombre}) tiene "
             f"un error del {num(error_despues)}% en {n:,} {cuando}, que ningún modelo vio al entrenar.".replace(f"{n:,}", f"{n:,}".replace(",", ".")))
    dif = error_despues - error_antes
    if ic[1] < 0:
        return frase + f" Mejora {num(-dif)} puntos (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    if ic[0] > 0:
        return frase + f" Es {num(dif)} puntos peor (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    return (frase + f" La diferencia ({num(dif)} puntos) está dentro del margen del azar (IC 95%: {num(ic[0])} a "
            f"{num(ic[1])}): los dos modelos rinden igual.")


def nombres_salida(modelo) -> list[str]:
    """Nombres de las columnas que realmente ve el modelo (después del one-hot)."""
    return list(modelo.named_steps["preparacion"].get_feature_names_out())


In [ ]:
%%writefile flujo.py
"""Paso común de los sprints 3 a 6: cargar las tablas planas y construir las características del Sprint 2.

Así todos los sprints usan exactamente las mismas columnas, aprendidas solo con el entrenamiento.
"""
from __future__ import annotations

import pandas as pd

import caracteristicas as ca
import datos


def cargar_y_construir(fuente: str = "auto") -> dict:
    """Devuelve los inscritos con características, las referencias del Sprint 2, el origen de los datos y las
    respuestas de la entrevista por sección (la entrada del modelo NLP; no son columnas del modelo de matrícula)."""
    import nlp_vocacion as nv
    inscritos, origen = datos.cargar_inscritos(fuente)
    ref = ca.ajustar_referencias(inscritos)
    return {"inscritos": ca.construir_caracteristicas(inscritos, ref), "referencias": ref, "origen": origen,
            "entrevista": inscritos[["inscripcion_id", *nv.SECCIONES]]}


def conjunto(d: pd.DataFrame, *nombres: str) -> pd.DataFrame:
    """Filtra por partición temporal: conjunto(d, 'ENTRENAMIENTO', 'VALIDACION')."""
    return d[d["particion"].isin(nombres)]


def modelos_finales(f: dict, parametros: dict) -> dict:
    """Reentrena los modelos finales del Sprint 3 con los hiperparámetros ya elegidos (los guarda modelos/*.json).

    Sirve en Google Colab, donde no están los archivos .joblib: con la misma semilla el resultado es idéntico.
    parametros = {"matricula": {"tipo", "parametros"}, "xgboost": {"parametros"}}
    """
    import modelado as mo
    cfg = ca.configuracion()
    semilla = cfg["modelado"]["semilla"]
    d, ref = f["inscritos"], f["referencias"]
    hist = conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
    V, V2 = mo.VARIABLES_MATRICULA_NLP, mo.VARIABLES_MATRICULA          # Sprint 3 (con NLP) · Sprint 2 (sin NLP)
    Xh, yh = mo.preparar_X(hist, ref, V), hist["matriculado"].astype(int)
    pm = parametros["matricula"]
    return {
        "referencias": ref,
        "modelo_matricula": mo.construir_pipeline(pm["tipo"], pm["parametros"], V, semilla).fit(Xh, yh),
        "modelo_matricula_xgboost": mo.construir_pipeline("xgboost", parametros["xgboost"]["parametros"], V, semilla).fit(Xh, yh),
        # La regresión logística del Sprint 2 (sin ajuste de hiperparámetros ni NLP), para compararla en igualdad de condiciones
        "logistica_sprint2": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V2, semilla).fit(
            mo.preparar_X(hist, ref, V2), yh),
        "fondo_matricula": Xh.sample(300, random_state=1),
    }


In [ ]:
%%writefile graficas_sprint2.py
"""Gráficas del Sprint 2 en español sencillo. Las usan ejecutar_sprint2.py (las guarda en PNG) y el cuaderno (las muestra).

Cada función devuelve la figura de matplotlib. Los ejes y títulos usan nombres claros (NOMBRES_LEGIBLES) en vez de los
nombres técnicos de las columnas, y las barras llevan su valor escrito para leerlas sin mirar el eje.
"""
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import caracteristicas as ca

AZUL, NARANJA, GRIS, VERDE = "#1B365D", "#E07B39", "#9AA5B1", "#2E7D5B"
COLORES_SEGMENTO = ["#791734", "#C9A227", "#E07B39", "#4A6FA5", "#2E7D5B"]      # S1 a S5
NOMBRE_CORTO = {"estrato": "estrato", "log_distancia": "distancia al campus", "punt_global": "puntaje ICFES",
                "edad_al_inscribirse": "edad",
                "colegio_privado": "colegio privado", "colegio_promedio_saber11": "nivel del colegio"}
ORDEN = {"segmento_rendimiento_icfes": ["BAJO", "MEDIO", "ALTO", "SUPERIOR"], "zona_distancia": ["CERCA", "MEDIA", "LEJOS"],
         "estrato_grupo": ["BAJO", "MEDIO", "ALTO"], "rezago_saber11": ["INMEDIATO", "UN_ANIO", "DOS_O_MAS"],
         "aplica_beca_mi_vocacion": ["No", "Sí"], "canal_agrupado": ["DIGITAL", "EVENTO", "RELACIONAL"]}


def eleccion_k(ref: dict) -> plt.Figure:
    km = ref["kmeans"]
    ev = pd.DataFrame(km["evaluacion"])
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    ejes[0].plot(ev["k"], ev["inercia"], "o-", color=AZUL)
    ejes[0].set(title="Método del codo: con más grupos, cada grupo es más parejo",
                xlabel="número de segmentos (k)", ylabel="dispersión dentro de los grupos (inercia)")
    ejes[1].plot(ev["k"], ev["silueta"], "o-", color=NARANJA)
    ejes[1].axvline(km["k"], color=GRIS, ls="--", label=f"se eligieron {km['k']} segmentos")
    ejes[1].set(title="Silueta: qué tan separados quedan los grupos (más alto = mejor)",
                xlabel="número de segmentos (k)", ylabel="silueta media (de −1 a 1)")
    ejes[1].legend(fontsize=9)
    for ax in ejes:
        ax.set_xticks(ev["k"])
    fig.tight_layout()
    return fig


def mapa_segmentos(d: pd.DataFrame, ref: dict) -> plt.Figure:
    """Cómo se ven los segmentos de k-means. k-means trabaja con 6 variables a la vez; para dibujarlas en un plano se
    proyectan en sus 2 componentes principales (PCA con las medias y escalas del entrenamiento). Cada punto es un
    inscrito, su color es el segmento que le asignó k-means y los círculos grandes son los centros de cada segmento."""
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    ent = (d["particion"] == "ENTRENAMIENTO").to_numpy()
    media = Z[ent].mean(axis=0)
    _, s, vt = np.linalg.svd(Z[ent] - media, full_matrices=False)
    comp = vt[:2].copy()
    for i in range(2):                                   # la variable con más peso apunta a la derecha / arriba
        if comp[i][np.argmax(np.abs(comp[i]))] < 0:
            comp[i] = -comp[i]
    var = (s ** 2 / (s ** 2).sum())[:2]
    P = (Z - media) @ comp.T
    C = (km["centros_z"].to_numpy(float) - media) @ comp.T          # centros, en el mismo orden que los nombres
    fig, ax = plt.subplots(figsize=(13.5, 7.6))
    for i, nombre in enumerate(km["nombres"]):
        m = (d["segmento_mercado"] == nombre).to_numpy()
        ax.scatter(P[m, 0], P[m, 1], s=16, alpha=0.35, color=COLORES_SEGMENTO[i % 5], edgecolors="none",
                   label=f"{nombre} ({format(int(m.sum()), ',').replace(',', '.')} inscritos)")
    for i, nombre in enumerate(km["nombres"]):
        ax.scatter(*C[i], s=900, color=COLORES_SEGMENTO[i % 5], edgecolors="white", linewidths=3, zorder=5)
        ax.annotate(nombre.split(":")[0], C[i], ha="center", va="center", color="white", fontsize=13,
                    fontweight="bold", zorder=6)

    def eje(i: int) -> str:
        cargas = pd.Series(comp[i], index=km["variables"])
        top = cargas.abs().sort_values(ascending=False).index[:2]
        partes = [("más " if cargas[v] > 0 else "menos ") + NOMBRE_CORTO.get(v, v) for v in top]
        return f"Dimensión {i + 1} ({var[i] * 100:.0f}% de la variación) → {' y '.join(partes)}"

    ax.set(xlabel=eje(0), ylabel=eje(1),
           title="Así se ven los segmentos de k-means: cada punto es un inscrito; el círculo grande, el centro del grupo")
    ley = ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=10, markerscale=2.2)
    for h in ley.legend_handles:
        h.set_alpha(1)
    ax.axhline(0, color=GRIS, lw=0.6, ls=":")
    ax.axvline(0, color=GRIS, lw=0.6, ls=":")
    fig.tight_layout()
    return fig


def perfil_segmentos(d: pd.DataFrame) -> plt.Figure:
    """Tabla de colores: cada celda muestra el valor REAL promedio del segmento; el color dice si está por encima
    (rojo) o por debajo (azul) del promedio de todos los inscritos."""
    t = ca.tabla_segmentos(d)
    cols = ["Puntaje ICFES", "Distancia por vías (km)", "Estrato promedio",
            "% de colegio público", "Edad (años)", "% que se matriculó"]
    valores = t[cols]
    seg, total = valores.drop(index="Todos los inscritos"), valores.loc["Todos los inscritos"]
    rel = (seg - total) / seg.std().replace(0, 1)
    fig, ax = plt.subplots(figsize=(13, 0.75 * len(valores) + 1.8))
    colores = np.vstack([rel.clip(-1.5, 1.5).to_numpy(float), np.zeros((1, len(cols)))])
    ax.imshow(colores, cmap="coolwarm", vmin=-2.2, vmax=2.2, aspect="auto")
    ax.set_xticks(range(len(cols)), [c.replace(" (", "\n(") for c in cols], fontsize=9)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(valores)), valores.index, fontsize=9)
    for (i, j), v in np.ndenumerate(valores.to_numpy(float)):
        texto = f"{v:.0f}%" if cols[j].startswith("%") else f"{v:.2f}" if "Notas" in cols[j] else \
            f"{v:.0f}" if cols[j] == "Puntaje ICFES" else f"{v:.1f}"
        ax.text(j, i, texto, ha="center", va="center", fontsize=10,
                fontweight="bold" if i == len(valores) - 1 else "normal")
    ax.axhline(len(valores) - 1.5, color="black", lw=1)
    ax.set_title("¿Cómo es cada segmento? Promedio de cada grupo  ·  rojo = más alto que el promedio · azul = más bajo",
                 fontsize=11, pad=12)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.tight_layout()
    return fig


def tamano_y_matricula(d: pd.DataFrame) -> plt.Figure:
    t = ca.tabla_segmentos(d).drop(index="Todos los inscritos").iloc[::-1]
    tasa_total = d["matriculado"].astype(float).mean() * 100
    fig, ejes = plt.subplots(1, 2, figsize=(13, 0.6 * len(t) + 1.6), sharey=True)
    b = ejes[0].barh(t.index, t["Inscritos"], color=AZUL)
    ejes[0].bar_label(b, labels=[f"{n:,}  ({p:.0f}%)".replace(",", ".") for n, p in zip(t["Inscritos"], t["% del total"])],
                      padding=3, fontsize=9)
    ejes[0].set(title="¿Cuántos inscritos hay en cada segmento?", xlabel="inscritos")
    ejes[0].set_xlim(0, t["Inscritos"].max() * 1.3)
    b = ejes[1].barh(t.index, t["% que se matriculó"], color=[VERDE if v >= tasa_total else NARANJA for v in t["% que se matriculó"]])
    ejes[1].bar_label(b, fmt="%.0f%%", padding=3, fontsize=9)
    ejes[1].axvline(tasa_total, color=GRIS, ls="--", label=f"todos los inscritos: {tasa_total:.0f}%")
    ejes[1].set(title="¿Qué porcentaje se matriculó?", xlabel="% que pagó la matrícula", xlim=(0, 70))
    ejes[1].legend(fontsize=9, loc="lower right")
    fig.tight_layout()
    return fig


def tabla_matricula_por_grupo(ent: pd.DataFrame, columnas: list[str], cfg: dict | None = None) -> pd.DataFrame:
    """Tabla larga: para cada característica y cada grupo, cuántos inscritos hay y qué % se matriculó."""
    etiq = ca.etiquetas_categorias(cfg)
    filas = []
    for c in columnas:
        g = ent.assign(m=ent["matriculado"].astype(float)).groupby(c)["m"].agg(["size", "mean"])
        g = g.reindex([o for o in ORDEN.get(c, g.index) if o in g.index])
        for codigo, (n, tasa) in g.iterrows():
            filas.append({"Característica": ca.NOMBRES_LEGIBLES.get(c, c), "Grupo": etiq.get(c, {}).get(codigo, codigo),
                          "Inscritos": int(n), "% que se matriculó": round(tasa * 100, 1)})
    return pd.DataFrame(filas)


def matricula_por_grupo(ent: pd.DataFrame, columnas: list[str], cfg: dict | None = None) -> plt.Figure:
    t = tabla_matricula_por_grupo(ent, columnas, cfg)
    tasa_total = ent["matriculado"].astype(float).mean() * 100
    filas = int(np.ceil(len(columnas) / 3))
    fig, ejes = plt.subplots(filas, 3, figsize=(16, 3.6 * filas))
    for ax, c in zip(ejes.flat, columnas):
        g = t[t["Característica"] == ca.NOMBRES_LEGIBLES.get(c, c)].iloc[::-1]
        b = ax.barh(g["Grupo"], g["% que se matriculó"],
                    color=[VERDE if v >= tasa_total else NARANJA for v in g["% que se matriculó"]])
        ax.bar_label(b, labels=[f"{v:.0f}%  (n = {n})" for v, n in zip(g["% que se matriculó"], g["Inscritos"])],
                     padding=3, fontsize=8)
        ax.axvline(tasa_total, color=GRIS, ls="--")
        ax.set_xlim(0, 85)
        ax.set_title(ca.NOMBRES_LEGIBLES.get(c, c), fontsize=11)
        ax.set_xlabel("% que se matriculó")
    for ax in list(ejes.flat)[len(columnas):]:
        ax.axis("off")
    fig.suptitle(f"¿Qué porcentaje se matricula en cada grupo? · inscritos de entrenamiento ({len(ent):,}) · verde = más "
                 f"que el promedio ({tasa_total:.0f}%, línea gris)".replace(",", "."), fontsize=12)
    fig.tight_layout()
    return fig


def palabras_nlp(claves: pd.DataFrame) -> plt.Figure:
    """Las palabras que más suben (verde) y más bajan (naranja) el puntaje vocacional en cada sección de la entrevista.
    claves: salida de nlp_vocacion.palabras_clave."""
    secciones = list(dict.fromkeys(claves["seccion"]))
    fig, ejes = plt.subplots(1, len(secciones), figsize=(15, 5.2))
    for ax, s in zip(ejes, secciones):
        c = claves[claves["seccion"] == s].sort_values("peso")
        ax.barh(c["termino"], c["peso"], color=[VERDE if p > 0 else NARANJA for p in c["peso"]])
        ax.axvline(0, color=GRIS, lw=0.8)
        ax.set(title=s, xlabel="peso en el puntaje (+ sube, − baja)")
        ax.tick_params(axis="y", labelsize=9)
    fig.suptitle("Las palabras que determinan la vocación según el modelo NLP (TF-IDF + Ridge, ajustado con el entrenamiento)",
                 fontsize=12)
    fig.tight_layout()
    return fig


def puntaje_nlp(d: pd.DataFrame, conjunto: str = "VALIDACION") -> plt.Figure:
    """Izquierda: el puntaje NLP frente al del entrevistador. Derecha: % que se matricula por quintil del puntaje NLP."""
    con = d[(d["particion"] == conjunto) & d["puntaje_vocacional_nlp"].notna()]
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4.4))
    grupos = [con.loc[con["puntaje_vocacional"] == k, "puntaje_vocacional_nlp"] for k in range(1, 6)]
    ejes[0].boxplot(grupos, patch_artist=True, boxprops={"facecolor": "#DCE3EC"}, medianprops={"color": AZUL, "lw": 2})
    ejes[0].set_xticks(range(1, 6), [str(k) for k in range(1, 6)])     # sin tick_labels: sirve con cualquier matplotlib
    ejes[0].plot(range(1, 6), range(1, 6), ls="--", color=GRIS, lw=1, label="lectura perfecta")
    ejes[0].set(title="¿Las palabras dicen lo mismo que el entrevistador?", xlabel="puntaje del entrevistador (1 a 5)",
                ylabel="puntaje NLP según sus palabras")
    ejes[0].legend(fontsize=9)
    quintil = pd.qcut(con["puntaje_vocacional_nlp"], 5, labels=["Q1 (más bajo)", "Q2", "Q3", "Q4", "Q5 (más alto)"])
    tasa = con["matriculado"].astype(float).groupby(quintil, observed=True).mean() * 100
    sin = d.loc[(d["particion"] == conjunto) & (d["entrevista_realizada"] == 0), "matriculado"].astype(float).mean() * 100
    barras = ejes[1].bar(list(tasa.index.astype(str)) + ["sin entrevista"], list(tasa) + [sin],
                         color=[AZUL] * len(tasa) + [GRIS])
    ejes[1].bar_label(barras, fmt="%.0f%%", fontsize=9)
    ejes[1].set(title="¿Qué porcentaje se matricula según el puntaje NLP?", ylabel="% que se matriculó",
                ylim=(0, max(list(tasa) + [sin]) * 1.2))
    ejes[1].tick_params(axis="x", labelsize=8)
    nombre = {"VALIDACION": "validación 2026-1", "ENTRENAMIENTO": "entrenamiento"}.get(conjunto, conjunto.lower())
    fig.suptitle(f"Puntaje vocacional NLP · {nombre} ({len(con):,} entrevistas)".replace(",", "."), fontsize=11)
    fig.tight_layout()
    return fig


def error_logistica(metr: pd.DataFrame) -> plt.Figure:
    """metr: índice = conjunto (VALIDACION, PRUEBA) con las columnas error_pct y auc_roc de la regresión logística."""
    conjuntos = {"VALIDACION": "Validación (2026-1)", "PRUEBA": "Prueba (2026-2)"}
    m = metr.rename(index=conjuntos).loc[list(conjuntos.values())]
    fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, titulo, fmt in [(ejes[0], "error_pct", "¿Cuántas veces se equivoca? (más bajo = mejor)", "{:.1f}%"),
                                 (ejes[1], "auc_roc", "¿Qué tan bien ordena? AUC (0,5 = azar; 1 = perfecto)", "{:.3f}")]:
        barras = ax.bar(m.index, m[col], color=[AZUL, NARANJA])
        ax.bar_label(barras, labels=[fmt.format(v) for v in m[col]], fontsize=9)
        ax.set(title=titulo, ylim=(0, m[col].max() * 1.25))
    ejes[0].set_ylabel("% de inscritos mal clasificados")
    fig.suptitle("Regresión logística del Sprint 2", fontsize=11)
    fig.tight_layout()
    return fig


In [ ]:
import sys
import warnings

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
EN_COLAB = True

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, GRIS = "#1B365D", "#E07B39", "#9AA5B1"

## 1. Datos y partición temporal
Los datos se dividen por **tiempo**, no al azar: se aprende con el pasado y se evalúa con el futuro, como pasará en la
realidad. La campaña 2027-1 no tiene resultado (se predice); su resultado real está sellado hasta el Sprint 5.

| Conjunto | Periodos | Para qué |
|---|---|---|
| ENTRENAMIENTO | 2024-1 a 2025-2 | Aprender escalas, k-means y modelos |
| VALIDACION | 2026-1 | Elegir hiperparámetros (Sprint 3) |
| PRUEBA | 2026-2 | Medir el error final, una sola vez |
| CAMPANA | 2027-1 | Lo que el modelo predice |

In [ ]:
import caracteristicas as ca
import flujo
import modelado as mo

# Lee las tablas planas por la API de Supabase (solo lectura): dataset_matricula y las respuestas de la entrevista
# (dataset_entrevista). Después construye las columnas del Sprint 2 (incluido el puntaje vocacional NLP).
f = flujo.cargar_y_construir("supabase" if EN_COLAB else "auto")
d, ref, cfg = f["inscritos"], f["referencias"], ca.configuracion()
print("Origen de los datos:", f["origen"])
print(f"Inscritos: {len(d):,} · con entrevista: {int(d['entrevista_realizada'].sum()):,}")
print(d["particion"].value_counts().to_string())

## 2. La entrevista de admisión en texto y el modelo NLP
La entrevista de admisión es **cualitativa**: el aspirante responde con sus palabras. Se usan tres secciones del
formulario de Unicervantes, con 13 preguntas abiertas:

| Sección | Preguntas |
|---|---|
| Interés por el programa académico | Por qué la carrera, defina el programa, qué le gusta, cómo se ve como profesional, otra carrera |
| Historia de vida | Sobre usted, ocupación del hogar, trabajo y financiación, cualidades, aspectos por mejorar, colegio, tiempo libre |
| Interés por la universidad | ¿Por qué quiere estudiar en Unicervantes? |

El entrevistador da un **puntaje vocacional de 1 a 5** (está en `dataset_matricula`). Un **modelo NLP** lee las
respuestas y aprende qué palabras acompañan a cada puntaje:

1. **Preprocesamiento:** minúsculas, sin tildes, sin palabras vacías ("de", "la", "que"…), pero conservando las que
   cambian el sentido ("no", "todavía", "ya"). Se forman palabras y pares de palabras dentro de cada frase.
2. **TF-IDF:** cada término pesa más si aparece mucho en esa entrevista y poco en las demás. Un TF-IDF por sección.
3. **Regresión Ridge:** aprende el peso de cada término para estimar el puntaje del entrevistador. Ridge castiga los
   pesos grandes (α elegido con validación cruzada **dentro** del entrenamiento).

El resultado, `puntaje_vocacional_nlp` (1 a 5), es la vocación según las palabras del propio aspirante. No copia al
entrevistador: las palabras también muestran lo que él no alcanza a ver, y por eso pueden ayudar a predecir la
matrícula (Sprint 3).

- **Sin fuga de datos:** el NLP se ajusta solo con el entrenamiento. Las entrevistas del entrenamiento reciben una
  predicción **fuera de pliegue** (validación cruzada de 5 pliegues): ninguna se puntúa con un modelo que ya la vio.
- **Salud mental:** no se registra ni entra al modelo; es un dato sensible (Ley 1581 de 2012, art. 5).

> No hay respuestas reales de entrevistas en datos abiertos: se simulan con un banco de frases en español (supuestos
> S-48 y S-49, `src/textos_entrevista.py`).

In [ ]:
import caracteristicas as ca
import nlp_vocacion as nv

print(f"Inscritos con entrevista: {int(d['entrevista_realizada'].sum()):,} de {len(d):,} (13 respuestas cada uno)")
preguntas = pd.DataFrame(cfg["generador"]["entrevista_texto"]["preguntas"])
display(preguntas.rename(columns={"id": "pregunta", "seccion": "sección", "texto": "pregunta del formulario"}))
textos = f["entrevista"]                     # las respuestas por sección: la entrada del NLP
ejemplo = textos[textos["texto_interes_programa"].notna()].iloc[0]
for col, nombre in nv.SECCIONES.items():
    print(f"\n{nombre}:\n  {ejemplo[col]}")
print("\nCómo lo ve el modelo (primera sección, primeros 20 términos):", nv.terminos(ejemplo["texto_interes_programa"])[:20])

### Qué aprendió el modelo NLP
Las palabras con peso positivo suben el puntaje vocacional; las de peso negativo lo bajan. Se mide en la validación
(2026-1), que el NLP nunca vio: error absoluto medio (MAE) frente al puntaje del entrevistador, comparado con predecir
siempre el promedio, y la correlación de Spearman (orden).

In [ ]:
import graficas_sprint2 as gr
from sklearn.metrics import roc_auc_score


def ver(fig):
    """Muestra la figura en su lugar (antes de las tablas que siguen) y la cierra."""
    display(fig)
    plt.close(fig)


nlp = ref["nlp"]
print(f"Entrevistas de entrenamiento: {nlp['n_entrevistas']:,} · términos: {nlp['n_terminos']:,} · α de Ridge: {nlp['alpha']}")
ver(gr.palabras_nlp(nv.palabras_clave(nlp)))
filas = {}
for conj in ["ENTRENAMIENTO", "VALIDACION"]:
    x = flujo.conjunto(d, conj).dropna(subset=["puntaje_vocacional_nlp"])
    filas[conj] = nv.metricas(x["puntaje_vocacional"], x["puntaje_vocacional_nlp"], nlp["media_entrenamiento"]) | {
        "AUC del NLP solo (matrícula)": roc_auc_score(x["matriculado"].astype(int), x["puntaje_vocacional_nlp"])}
display(pd.DataFrame(filas).T.round(3))
ver(gr.puntaje_nlp(d, "VALIDACION"))
v = filas["VALIDACION"]
print(f"En la validación el NLP se equivoca {v['mae']:.2f} puntos en promedio frente al entrevistador "
      f"(predecir siempre el promedio: {v['mae_promedio']:.2f}).")

## 3. Lo que se aprende solo del entrenamiento
Si las escalas o los grupos se calcularan con todos los datos, la validación y la prueba "verían" información del
futuro (fuga de datos) y el error saldría engañosamente bajo. Por eso `ajustar_referencias` usa **solo** el
entrenamiento: la media y la desviación del Saber 11 (para su término cuadrático), los grupos de k-means y el modelo NLP.

In [ ]:
print("Estandarización (media y desviación del entrenamiento):")
display(pd.DataFrame(ref["estandarizacion"], index=["media", "desviación"]).round(3))

## 4. Las columnas nuevas
Cada columna responde una pregunta de negocio. Las que dependen de una regla (Sí/No de becas, bandas del ICFES,
valor neto) se calculan igual para todos; las que dependen de datos (k-means y el puntaje NLP) usan lo aprendido en
el entrenamiento.

In [ ]:
print(f"Columnas nuevas: {len(ca.NUEVAS)}")
display(pd.DataFrame({"columna": list(ca.DICCIONARIO), "qué es": list(ca.DICCIONARIO.values())}))
muestra = d[["inscripcion_id", "periodo", "punt_global", "aplica_beca_saber11", "aplica_beca_mi_vocacion",
             "puntaje_vocacional", "puntaje_vocacional_nlp", "segmento_rendimiento_icfes",
             "valor_neto_primer_semestre", "ahorro_beca_primer_semestre", "zona_distancia", "segmento_mercado"]]
display(muestra.head(8))
print("¿Aplica a Mi Vocación?", d["aplica_beca_mi_vocacion"].value_counts().to_dict(),
      "· ¿Aplica a Saber 11?", d["aplica_beca_saber11"].value_counts().to_dict())
print("Nulos en las columnas nuevas:", int(d[ca.NUEVAS].drop(columns="puntaje_vocacional_nlp").isna().sum().sum()),
      "· puntaje NLP vacío solo sin entrevista:", int(d["puntaje_vocacional_nlp"].isna().sum()))

## 5. Segmentación de mercado con k-means
k-means agrupa a los aspirantes por parecido en su perfil (estrato, distancia, ICFES, edad y tipo de colegio). No se
usan la media ICFES del colegio (Admisiones no suele tener ese dato) ni las notas del colegio (dejaron de ser variables
del análisis). Antes se **estandarizan** las variables (restar la media y dividir por la desviación), para que ninguna pese
más solo por estar en otra escala. El número de grupos k se elige con la **silueta**: qué tan cerca está cada aspirante
de su grupo frente al grupo vecino (de −1 a 1; más alto es mejor).

In [ ]:
ver(gr.eleccion_k(ref))
print("Mapa de los segmentos: las 6 variables proyectadas en 2 dimensiones (componentes principales).")
ver(gr.mapa_segmentos(d, ref))
print("Cada segmento se nombra por los 2 rasgos en que más se aleja del promedio de los inscritos.")
ver(gr.perfil_segmentos(d))
ver(gr.tamano_y_matricula(d))
print("Tabla de segmentos (promedios reales de cada grupo):")
display(ca.tabla_segmentos(d))

**Cómo leer el mapa.** k-means usa 5 variables a la vez, así que no se puede dibujar directamente. El mapa las resume en
2 dimensiones (componentes principales): cada punto es un inscrito, el color es el segmento que le asignó k-means y el
círculo grande es el centro del grupo. Los grupos se tocan en los bordes: por eso la silueta es baja.

**Cómo leer los nombres.** Cada segmento se nombra con los **2 rasgos en que más se diferencia del promedio**
de todos los inscritos. Los nombres son **relativos**: "viven cerca del campus" quiere decir más cerca que el promedio
(15,6 km por vías), no a menos de cierta distancia; "colegio público" quiere decir que casi todos vienen de colegios
oficiales (públicos). La tabla muestra los valores reales para comprobarlo.

| Segmento | En palabras sencillas |
|---|---|
| S1: Colegio privado, estrato más alto | El grupo más grande: colegios privados, estratos más altos y el mejor ICFES (muchos aplican a la beca Saber 11); el que más se matricula |
| S2: Colegio público, viven cerca del campus | Egresados de colegios públicos que viven relativamente cerca (unos 10 km por vías) |
| S3: Viven lejos del campus, estrato más bajo | Viven a unos 29 km por vías y son de estratos 1 y 2; el que menos se matricula |
| S4: Aspirantes mayores, puntaje ICFES bajo | Unos 22 años en promedio: no vienen directo del colegio |

**Qué tan buenos son los grupos.** La silueta es baja (cerca de 0,24): los aspirantes no forman grupos muy separados,
algo común en datos de personas. Aun así, los segmentos tienen perfiles con sentido de negocio y tasas de matrícula
distintas (del 49% al 61%), por eso sirven para orientar la estrategia de captación (Sprint 5).

## 6. Relación de las columnas nuevas con la matrícula (entrenamiento)

In [ ]:
ent = flujo.conjunto(d, "ENTRENAMIENTO")
columnas = ["segmento_rendimiento_icfes", "aplica_beca_mi_vocacion", "canal_agrupado", "zona_distancia", "estrato_grupo",
            "rezago_saber11"]
ver(gr.matricula_por_grupo(ent, columnas))
display(gr.tabla_matricula_por_grupo(ent, columnas))
correl = ent[["puntaje_vocacional", "puntaje_vocacional_nlp", "valor_neto_primer_semestre", "log_distancia"]].corrwith(
    ent["matriculado"].astype(float))
display(correl.rename(ca.DICCIONARIO).round(3).rename("correlación con la matrícula").to_frame())

**Lectura.**
- El segmento ICFES ALTO (310-349) se matricula más que el SUPERIOR (≥ 350): los puntajes muy altos tienen más
  opciones puesto a que las universidades los buscan. Es la "U invertida" que el modelo deberá captar.
- Quien aplica a Mi Vocación se matricula más: la beca baja el precio.
- Los canales relacionales (referido, inmersión, convenio) convierten más que los eventos y lo digital.
- La matrícula baja con la distancia al campus (por vías, desde la dirección de cada inscrito).
- Quien presentó la entrevista se matricula más que quien no la presentó. Entre los entrevistados, el puntaje NLP se
  relaciona con la matrícula más que el puntaje del entrevistador: las palabras muestran algo más que su calificación.

## 7. Regresión logística: el primer modelo
La regresión logística estima la **probabilidad** de que un inscrito se matricule a partir de todas sus variables:

`log(p / (1 − p)) = β₀ + β₁x₁ + … + βₖxₖ`

Cada β dice cuánto sube o baja la probabilidad cuando la variable cambia. Las variables numéricas se estandarizan y las
categóricas se codifican con *one-hot* (una columna 0/1 por categoría), dentro del mismo modelo.

- Se entrena solo con el **entrenamiento** (2024-1 a 2025-2) y **sin ajustar hiperparámetros**: C = 1, fijado antes de
  ver resultados (`config/reglas_negocio.yaml → modelado.sprint2_logistica`). El ajuste se hace en el Sprint 3.
- Se predice "se matricula" si la probabilidad es ≥ 50%. El **error %** es el porcentaje de inscritos mal clasificados.
- El **AUC** mide qué tan bien ordena a los inscritos: 0,5 es azar y 1 es perfecto.
- La entrevista entra con el **puntaje del entrevistador** (0 si no hubo entrevista). El puntaje NLP se agrega en el
  Sprint 3, para medir cuánto mejora el modelo.
- Es el **modelo base** del proyecto: el Sprint 3 lo compara con modelos más avanzados.

In [ ]:
V = mo.VARIABLES_MATRICULA
ent = flujo.conjunto(d, "ENTRENAMIENTO")
logistica = mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V, cfg["modelado"]["semilla"]).fit(
    mo.preparar_X(ent, ref, V), ent["matriculado"].astype(int))
coef = pd.Series(logistica.named_steps["modelo"].coef_[0], index=logistica.named_steps["preparacion"].get_feature_names_out())
print("Los 10 coeficientes más grandes (variables estandarizadas; positivo = sube la probabilidad):")
display(coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3).rename("β").to_frame())
resultados = {}
for conj in ["VALIDACION", "PRUEBA"]:
    X = flujo.conjunto(d, conj)
    resultados[conj] = mo.metricas(X["matriculado"], logistica.predict_proba(mo.preparar_X(X, ref, V))[:, 1])
metr = pd.DataFrame(resultados).T.astype(float)
display(metr[["n", "error_pct", "auc_roc", "brier"]].round(3))
ver(gr.error_logistica(metr))
b = resultados["PRUEBA"]
print(f"\nLa regresión logística se equivoca en el {b['error_pct']:.1f}% de los inscritos de 2026-2 (AUC {b['auc_roc']:.3f}). "
      "Este es el punto de comparación del Sprint 3.")